# Lake Victoria Fishery: Maximum Sustainable Yield (MSY) Notebook

This notebook turns the MSY summary into an executable workflow using the Schaefer Surplus Production Model, focused on **Nile perch (*Lates niloticus*)**.

## 1) Mathematical Model: Schaefer Surplus Production

The model assumes logistic stock growth and defines MSY as:

$$MSY = \frac{rK}{4}$$

Where:
- $r$: intrinsic population growth rate
- $K$: carrying capacity

Fishing effort at MSY is:

$$E_{MSY} = \frac{r}{2q}$$

Where $q$ is the catchability coefficient.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Support execution from either repo root or notebooks/ directory.
if not (Path.cwd() / "msy_plotting.py").exists() and (Path.cwd().parent / "msy_plotting.py").exists():
    sys.path.insert(0, str(Path.cwd().parent))

from msy_math import msy, e_msy, build_scenario_table, build_catch_table
from msy_plotting import (
    configure_plot_style,
    make_tonnes_formatter,
    plot_msy_sensitivity,
    plot_yield_effort,
    plot_catch_dashboard,
)

configure_plot_style()
tonnes_formatter = make_tonnes_formatter()

# Representative baseline values for Nile perch
r_baseline = 0.425
K_baseline = 2_030_000
q_baseline = 2.5e-7

print(f"Baseline MSY: {msy(r_baseline, K_baseline):,.0f} tonnes/year")
print(f"Baseline E_MSY: {e_msy(r_baseline, q_baseline):,.0f} effort units/year")

## 2) Data Needed to Compute MSY

For a full assessment year, combine these streams:

| Data Type | Description | Typical Source/Method |
|---|---|---|
| Annual Yield ($Y$) | Total catch (tonnes) by species across Kenya/Tanzania/Uganda | Catch Assessment Surveys (CAS) |
| Fishing Effort ($E$) | Number of boats, fishers, or boat-days | Frame Surveys (FS) |
| CPUE | Catch Per Unit Effort ($Y/E$) | Derived from CAS + FS |
| Biomass ($B$) | Total fish biomass in the lake | Hydroacoustic Surveys (HAS) |

## 3) Representative Nile Perch Parameter Range

Using the ranges in your summary:
- $K$ in [1,900,000, 2,160,000] tonnes
- $r$ in [0.40, 0.45] per year

In [ ]:
K_values = np.array([1_900_000, 2_160_000])
r_values = np.array([0.40, 0.45])

scenario_df = build_scenario_table(r_values, K_values)
scenario_df

In [ ]:
# Sensitivity surface over the parameter ranges
K_grid = np.linspace(1_900_000, 2_160_000, 100)
r_grid = np.linspace(0.40, 0.45, 100)
MSY_grid = np.array([[msy(r, K) for K in K_grid] for r in r_grid])

plot_msy_sensitivity(
    K_grid,
    r_grid,
    MSY_grid,
    r_baseline=r_baseline,
    k_baseline=K_baseline,
    tonnes_formatter=tonnes_formatter,
)

## 3b) Yield-Effort Frontier (Schaefer Curve)

This plot shows the equilibrium yield response to fishing effort and highlights the MSY peak.

In [ ]:
# Schaefer equilibrium yield curve: Y(E) = qKE - (q^2 K / r) E^2
E_star = e_msy(r_baseline, q_baseline)
E = np.linspace(0, 2 * E_star, 400)
Y = q_baseline * K_baseline * E - ((q_baseline ** 2) * K_baseline / r_baseline) * (E ** 2)
Y = np.clip(Y, 0, None)

plot_yield_effort(
    effort=E,
    yield_tonnes=Y,
    effort_msy=E_star,
    msy_tonnes=msy(r_baseline, K_baseline),
    tonnes_formatter=tonnes_formatter,
)

## 4) Current Trends and Risks to Interpret With MSY

Use model outputs together with management indicators:
- Fishing mortality near/above $F_{MSY}$ indicates overfishing risk.
- Juvenile-dominated catches reduce recovery potential and future yield.
- Eutrophication and water hyacinth can shift both $K$ and $q$, so re-estimation should be periodic.

## 5) Sample Country-Level Calculation

Replace the values below with historical records for one country and species.

This example uses a simple ratio against a reference MSY target to classify pressure.

In [ ]:
# Example structure: annual catches for one stock segment
country = "Uganda"
years = np.array([2019, 2020, 2021, 2022, 2023])
catch_tonnes = np.array([108000, 102500, 97000, 92000, 89000])

# Reference MSY target (edit as needed for your specific stock segment)
msy_reference = 86_000

result = build_catch_table(years, catch_tonnes, msy_reference)
print(f"Country: {country}")
result

In [ ]:
plot_catch_dashboard(
    result=result,
    country=country,
    msy_reference=msy_reference,
    tonnes_formatter=tonnes_formatter,
)

## 6) Next Step: Use Real CAS/FS/HAS Inputs

When you have actual annual observations, replace the example arrays with your country-specific data and re-run all cells.